# Workspace Lakehouse and Warehouse Audit

This notebook reads the copied workbook, ignores blank Keep values, annotates dependency evidence in column G, and evaluates only `NO` rows for guarded deletion. The default is a dry run.

In [ ]:
from pathlib import Path
import json
import sys
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from audit_cleanup import analyze_workbook, annotate_column_g, execute_decisions

In [ ]:
repo = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
input_path = repo / 'data' / 'Workspace_LH_and_DW_Audit.xlsx'
annotated_path = repo / 'data' / 'Workspace_LH_and_DW_Audit_annotated.xlsx'
results_path = repo / 'data' / 'cleanup_results.json'
workbook, decisions, metadata = analyze_workbook(input_path)
annotate_column_g(workbook, decisions, metadata)
workbook.save(annotated_path)
print(metadata)
print('NO rows:', len(decisions))
print('eligible before live resolution:', sum(d.eligible_for_deletion for d in decisions))

In [ ]:
# Read-only live catalog resolution. No deletion is performed in this cell.
results = execute_decisions(decisions, execute=False)
results_path.write_text(json.dumps(results, indent=2), encoding='utf-8')
for result in results:
    print(result['workspace'], result['item'], result['action'], '=>', result['live_action'])

## Controlled deletion gate

Do not enable deletion until every eligible row has an exact live workspace and item identity. The workbook currently names `ws_prod_legacy`; the notebook must not guess that this is another workspace. If the exact workspace is resolved and deletion is approved, set `FABRIC_ALLOW_DELETIONS=1`, set `FABRIC_DELETE_CONFIRMATION=DELETE_APPROVED_NO_DEPENDENCY_ITEMS`, and run `execute_decisions(decisions, execute=True)`.